# LeetCode 4051 — Count Subarrays with Distant Sums (Hard)

Bài toán LeetCode:

```python
class Solution:
    def distantSubarrays(self, nums: List[int], goal: int, k: int) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4051 (Biweekly Contest 191 — câu 4, rating khoảng 2139).

Đây là bài tập **rất kinh điển** để luyện bộ ba kỹ thuật:

- **Tổng tiền tố (prefix sum):** tổng đoạn = hiệu hai tiền tố.
- **Đếm phần bù:** đếm đoạn "gần" (dễ đếm hơn) rồi lấy tổng trừ đi.
- **Cấu trúc dữ liệu đếm theo khoảng giá trị:** danh sách sắp xếp + `bisect`, cây Fenwick (BIT) với nén tọa độ, hoặc chia để trị kiểu merge sort.

Mảng có **số âm**, nên kỹ thuật hai con trỏ (sliding window) **không** dùng được.

---

## Sau notebook này, bạn sẽ nắm được

1. Biến điều kiện trên tổng đoạn thành điều kiện trên **cặp tiền tố**.
2. Đếm phần bù: `|x| >= k` ⇔ không phải `|x| < k`.
3. Đếm "bao nhiêu tiền tố trước đó nằm trong đoạn `[lo, hi]`" bằng 3 cách: `bisect` + `insort`, BIT + nén tọa độ, merge sort.
4. Trường hợp đặc biệt `k = 0`.
5. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho mảng số nguyên `nums` (có thể âm), số nguyên `goal` và `k >= 0`. Một mảng con (đoạn liên tiếp, khác rỗng) là **xa** nếu

```text
| tổng(mảng con) - goal | >= k
```

Đếm số mảng con xa.

---

## Ví dụ 1

```python
nums = [1, 2, 1], goal = 4, k = 1   ->   5
```

Có 6 mảng con; chỉ `[1, 2, 1]` có tổng `4` với `|4 - 4| = 0 < 1` → không xa. Đáp án `6 - 1 = 5`.

## Ví dụ 2

```python
nums = [2, -1, 3], goal = 2, k = 2   ->   2
```

| mảng con | tổng | `|tổng - 2|` | xa? |
|---|---:|---:|:---:|
| `[2]` | 2 | 0 | |
| `[-1]` | -1 | 3 | ✔ |
| `[3]` | 3 | 1 | |
| `[2, -1]` | 1 | 1 | |
| `[-1, 3]` | 2 | 0 | |
| `[2, -1, 3]` | 4 | 2 | ✔ |

## Ví dụ 3

```python
nums = [-3, 1, 2], goal = 0, k = 3   ->   2      # hai mảng con xa: [-3] và [1, 2]
```

Tổng các mảng con: `[-3]=-3`, `[1]=1`, `[2]=2`, `[-3,1]=-2`, `[1,2]=3`, `[-3,1,2]=0`. Có `|−3| >= 3` và `|3| >= 3` → **2**.

---

## Ràng buộc

```text
1 <= nums.length <= 10^5
-10^9 <= nums[i], goal <= 10^9
0 <= k <= 10^9
```

# 2. Kiến thức nền tảng

## 2.1. Tổng tiền tố

`P[0] = 0`, `P[t] = nums[0] + ... + nums[t-1]`. Tổng đoạn `nums[i .. j-1]` = `P[j] - P[i]` với `0 <= i < j <= n`.

In [1]:
from typing import List
from itertools import accumulate

nums = [2, -1, 3]
P = [0] + list(accumulate(nums))
print(P)
print("tổng nums[1..2] =", P[3] - P[1])

[0, 2, 1, 4]
tổng nums[1..2] = 2


## 2.2. `bisect`: đếm số phần tử trong `[lo, hi]` của list đã sắp

`bisect_right(a, hi) - bisect_left(a, lo)`.

In [2]:
from bisect import bisect_left, bisect_right, insort

a = [0, 1, 1, 3, 5]
print(bisect_right(a, 3) - bisect_left(a, 1))     # các phần tử 1, 1, 3
insort(a, 2)
print(a)

3
[0, 1, 1, 2, 3, 5]


## 2.3. Biến đổi điều kiện

Đoạn ứng với cặp `(i, j)` **gần** (không xa) khi

```text
|P[j] - P[i] - goal| < k
⇔  goal - k < P[j] - P[i] < goal + k
⇔  P[j] - goal - k < P[i] < P[j] - goal + k
⇔  P[i] ∈ [P[j] - goal - k + 1,  P[j] - goal + k - 1]     (vì toàn số nguyên)
```

Vậy: duyệt `j` từ trái sang phải, **đếm số tiền tố `P[i]` (i < j) nằm trong một đoạn giá trị** phụ thuộc `P[j]`. Đây là bài toán đếm trên khoảng giá trị quen thuộc.

# 3. Cách 1 — Brute Force `O(n²)` với tổng tiền tố

(Không cần `O(n³)`: tính tổng đoạn tăng dần khi kéo dài đầu phải.)

In [3]:
def distant_bruteforce(nums: List[int], goal: int, k: int) -> int:
    n = len(nums)
    count = 0
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            if abs(s - goal) >= k:
                count += 1
    return count


print(distant_bruteforce([1, 2, 1], 4, 1), distant_bruteforce([2, -1, 3], 2, 2), distant_bruteforce([-3, 1, 2], 0, 3))

5 2 2


`n = 10^5` → `5·10^9` phép cộng — quá chậm.

# 4. Cách 2 — Đếm phần bù + danh sách sắp xếp (`bisect` + `insort`)

```text
đáp án = n(n+1)/2  -  số cặp (i < j) "gần"
```

Duyệt `P[0], P[1], ..., P[n]`; với mỗi `v = P[j]`, đếm trong danh sách các tiền tố đã thấy những giá trị thuộc `[v - goal - k + 1, v - goal + k - 1]`, rồi chèn `v` vào.

## Mô phỏng ví dụ 2 (`P = [0, 2, 1, 4]`, `goal = 2`, `k = 2` → đoạn `[v - 3, v - 1]`)

| v = P[j] | đoạn cần đếm | danh sách trước | đếm "gần" | danh sách sau |
|---:|---|---|---:|---|
| 0 | `[-3, -1]` | `[]` | 0 | `[0]` |
| 2 | `[-1, 1]` | `[0]` | 1 (đoạn `[2]`, tổng 2) | `[0, 2]` |
| 1 | `[-2, 0]` | `[0, 2]` | 1 (đoạn `[2, -1]`, tổng 1) | `[0, 1, 2]` |
| 4 | `[1, 3]` | `[0, 1, 2]` | 2 (`[3]` tổng 3, `[-1, 3]` tổng 2) | `[0, 1, 2, 4]` |

Gần = 4 → đáp án `6 - 4 = 2` ✔.

In [4]:
def distant_sorted_list(nums: List[int], goal: int, k: int, verbose: bool = False) -> int:
    n = len(nums)
    total = n * (n + 1) // 2
    if k == 0:
        return total                      # |x| >= 0 luôn đúng
    seen = []
    close = 0
    prefix = 0
    for t in range(n + 1):
        if t > 0:
            prefix += nums[t - 1]
        lo, hi = prefix - goal - k + 1, prefix - goal + k - 1
        c = bisect_right(seen, hi) - bisect_left(seen, lo)
        if verbose:
            print(f"v={prefix:3} đoạn=[{lo}, {hi}] seen={seen} gần+={c}")
        close += c
        insort(seen, prefix)
    return total - close


print(distant_sorted_list([2, -1, 3], 2, 2, verbose=True))

v=  0 đoạn=[-3, -1] seen=[] gần+=0
v=  2 đoạn=[-1, 1] seen=[0] gần+=1
v=  1 đoạn=[-2, 0] seen=[0, 2] gần+=1
v=  4 đoạn=[1, 3] seen=[0, 1, 2] gần+=2
2


`bisect` là `O(log n)` nhưng `insort` phải **dời phần tử** trong list → `O(n)` mỗi lần, tệ nhất `O(n²)`. Trong thực tế `insort` rất nhanh (dời bộ nhớ bằng C) nên cách này thường vẫn qua được với `n = 10^5`; tuy vậy ta nên biết cách `O(n log n)` thật sự.

# 5. Cách 3 — Cây Fenwick (BIT) + nén tọa độ

Giá trị tiền tố lên tới `10^14` — không làm mảng đếm trực tiếp được. **Nén tọa độ:** sắp các giá trị `P` khác nhau, mỗi giá trị ↦ thứ hạng `0..m-1`. Đoạn `[lo, hi]` ↦ đoạn thứ hạng `[bisect_left(vals, lo), bisect_right(vals, hi))`.

Cây Fenwick hỗ trợ: `add(pos)` (thêm 1 tại thứ hạng `pos`) và `query(r)` = số phần tử có thứ hạng `< r`, cả hai `O(log m)`.

In [5]:
class Fenwick:
    def __init__(self, m: int):
        self.m = m
        self.tree = [0] * (m + 1)

    def add(self, pos: int) -> None:          # pos: 0-indexed
        i = pos + 1
        while i <= self.m:
            self.tree[i] += 1
            i += i & -i

    def query(self, r: int) -> int:           # số phần tử có thứ hạng < r
        s = 0
        while r > 0:
            s += self.tree[r]
            r -= r & -r
        return s


fw = Fenwick(5)
for pos in [0, 2, 2, 4]:
    fw.add(pos)
print([fw.query(r) for r in range(6)])

[0, 1, 1, 3, 3, 4]


In [6]:
def distant_fenwick(nums: List[int], goal: int, k: int) -> int:
    n = len(nums)
    total = n * (n + 1) // 2
    if k == 0:
        return total
    P = [0] + list(accumulate(nums))
    vals = sorted(set(P))
    fw = Fenwick(len(vals))
    close = 0
    for v in P:
        lo, hi = v - goal - k + 1, v - goal + k - 1
        close += fw.query(bisect_right(vals, hi)) - fw.query(bisect_left(vals, lo))
        fw.add(bisect_left(vals, v))
    return total - close


print(distant_fenwick([1, 2, 1], 4, 1), distant_fenwick([2, -1, 3], 2, 2), distant_fenwick([-3, 1, 2], 0, 3))

5 2 2


# 6. Cách 4 — Chia để trị kiểu Merge Sort

Ý tưởng của *Count of Range Sum* (LeetCode 327): đếm cặp `(i < j)` với `P[j] - P[i] ∈ [L, R]` (ở đây `L = goal - k + 1`, `R = goal + k - 1`).

- Chia mảng `P` làm đôi; đếm đệ quy trong mỗi nửa.
- Cặp chéo: `i` ở nửa trái, `j` ở nửa phải. Khi **hai nửa đã sắp xếp**, với mỗi `P[j]` tăng dần, các `P[i]` thỏa `P[j] - R <= P[i] <= P[j] - L` tạo một cửa sổ **trượt về một phía** → hai con trỏ, `O(độ dài)`.
- Trộn hai nửa đã sắp như merge sort.

Tổng `O(n log n)`, không cần nén tọa độ.

In [7]:
def distant_merge_sort(nums: List[int], goal: int, k: int) -> int:
    n = len(nums)
    total = n * (n + 1) // 2
    if k == 0:
        return total
    L, R = goal - k + 1, goal + k - 1
    P = [0] + list(accumulate(nums))

    def solve(arr):
        if len(arr) <= 1:
            return arr, 0
        mid = len(arr) // 2
        left, c1 = solve(arr[:mid])
        right, c2 = solve(arr[mid:])
        cnt = c1 + c2
        lo = hi = 0
        for x in right:                      # x = P[j], tăng dần
            while lo < len(left) and left[lo] < x - R:
                lo += 1
            while hi < len(left) and left[hi] <= x - L:
                hi += 1
            cnt += hi - lo
        merged, a, b = [], 0, 0              # trộn hai nửa đã sắp
        while a < len(left) and b < len(right):
            if left[a] <= right[b]:
                merged.append(left[a]); a += 1
            else:
                merged.append(right[b]); b += 1
        merged += left[a:] + right[b:]
        return merged, cnt

    return total - solve(P)[1]


print(distant_merge_sort([1, 2, 1], 4, 1), distant_merge_sort([2, -1, 3], 2, 2), distant_merge_sort([-3, 1, 2], 0, 3))

5 2 2


Vì sao hai con trỏ `lo`, `hi` chỉ tiến? Khi `x` tăng, cả hai mút `x - R` và `x - L` đều tăng, nên cửa sổ hợp lệ trong `left` (đã sắp) chỉ trượt sang phải.

# 7. Phiên bản đúng format LeetCode

Dùng Fenwick (Cách 3).

In [8]:
from bisect import bisect_left, bisect_right
from typing import List


class Solution:
    def distantSubarrays(self, nums: List[int], goal: int, k: int) -> int:
        n = len(nums)
        total = n * (n + 1) // 2
        if k == 0:
            return total
        P = [0]
        for x in nums:
            P.append(P[-1] + x)
        vals = sorted(set(P))
        m = len(vals)
        tree = [0] * (m + 1)

        def add(pos):
            i = pos + 1
            while i <= m:
                tree[i] += 1
                i += i & -i

        def query(r):
            s = 0
            while r > 0:
                s += tree[r]
                r -= r & -r
            return s

        close = 0
        for v in P:
            lo, hi = v - goal - k + 1, v - goal + k - 1
            close += query(bisect_right(vals, hi)) - query(bisect_left(vals, lo))
            add(bisect_left(vals, v))
        return total - close

In [9]:
sol = Solution()
tests = [
    ([1, 2, 1], 4, 1, 5, "ví dụ 1"),
    ([2, -1, 3], 2, 2, 2, "ví dụ 2"),
    ([-3, 1, 2], 0, 3, 2, "ví dụ 3"),
    ([5, 5], 7, 0, 3, "k = 0: mọi đoạn đều xa"),
    ([1, 1, 1], 2, 1, 4, "chỉ bỏ các đoạn tổng đúng bằng goal"),
    ([10**9] * 3, -10**9, 10**9, 6, "số lớn"),
]
for nums, goal, k, expected, name in tests:
    result = sol.distantSubarrays(nums, goal, k)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:38} -> {result}")

PASS | ví dụ 1                                -> 5
PASS | ví dụ 2                                -> 2
PASS | ví dụ 3                                -> 2
PASS | k = 0: mọi đoạn đều xa                 -> 3
PASS | chỉ bỏ các đoạn tổng đúng bằng goal    -> 4
PASS | số lớn                                 -> 6


Test `[1, 1, 1]`, `goal = 2`, `k = 1`: "gần" ⇔ `|tổng - 2| < 1` ⇔ tổng = 2: có 2 đoạn `[1, 1]` → `6 - 2 = 4`.

# 8. Vì sao đúng?

1. Mỗi mảng con ↔ đúng một cặp `(i, j)`, `0 <= i < j <= n`; tổng của nó là `P[j] - P[i]`.
2. Xa và gần là phần bù của nhau, nên `xa = n(n+1)/2 - gần`.
3. Gần ⇔ `P[i]` thuộc đoạn số nguyên `[P[j] - goal - k + 1, P[j] - goal + k - 1]` (biến đổi tương đương ở mục 2.3).
4. Khi xử lý `P[j]`, cấu trúc dữ liệu chứa đúng các `P[0..j-1]`, nên đếm đúng các `i < j`.
5. `k = 0`: không có đoạn gần (`|x| < 0` vô lý). Đoạn số nguyên ở bước 3 khi đó "ngược" (`lo > hi`), nên phải xử lý riêng (xem Lỗi 2).

# 9. Độ phức tạp

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Brute Force | mọi đoạn | `O(n²)` | `O(1)` |
| 2. bisect + insort | danh sách sắp | `O(n log n)` tìm + `O(n²)` chèn (tệ nhất) | `O(n)` |
| 3. Fenwick + nén | đếm theo thứ hạng | `O(n log n)` | `O(n)` |
| 4. Merge sort | đếm cặp chéo bằng 2 con trỏ | `O(n log n)` | `O(n)` |

In [10]:
import time
import random

random.seed(4051)
big = [random.randint(-10**9, 10**9) for _ in range(10**5)]
for name, fn in [("Fenwick", sol.distantSubarrays), ("bisect+insort", distant_sorted_list), ("merge sort", distant_merge_sort)]:
    start = time.time()
    fn(big, 12345, 10**9)
    print(f"{name:14}: {time.time() - start:.2f} giây")

Fenwick       : 0.37 giây


bisect+insort : 0.97 giây


merge sort    : 0.43 giây


# 10. Đối chiếu bốn cách

In [11]:
random.seed(51)
for _ in range(3000):
    nums = [random.randint(-5, 5) for _ in range(random.randint(1, 9))]
    goal = random.randint(-6, 6)
    k = random.randint(0, 5)
    expected = distant_bruteforce(nums, goal, k)
    assert distant_sorted_list(nums, goal, k) == expected
    assert distant_fenwick(nums, goal, k) == expected
    assert distant_merge_sort(nums, goal, k) == expected
    assert sol.distantSubarrays(nums, goal, k) == expected
print("Bốn cách khớp brute force trên 3000 test")

Bốn cách khớp brute force trên 3000 test


# 11. Những lỗi dễ mắc

## Lỗi 1 — Dùng hai con trỏ (sliding window)

Hai con trỏ chỉ đúng khi tổng đoạn **đơn điệu** theo độ dài — tức mọi phần tử không âm. Ở đây có số âm. Thử với phần "tổng `>= goal + k`" (một nửa của điều kiện xa): cửa sổ trượt đúng với mảng không âm nhưng sai ngay khi có số âm.

In [12]:
def wrong_sliding_at_least(nums, T):
    # Đếm mảng con có tổng >= T bằng cửa sổ trượt — chỉ đúng khi mọi phần tử >= 0
    count, left, s = 0, 0, 0
    for right in range(len(nums)):
        s += nums[right]
        while left <= right and s - nums[left] >= T:   # bỏ bớt đầu trái nếu vẫn đủ
            s -= nums[left]
            left += 1
        if s >= T:
            count += left + 1                           # mọi đầu trái 0..left đều đủ (giả định sai!)
    return count


def brute_at_least(nums, T):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i, n) if sum(nums[i:j + 1]) >= T)


print("Không âm [1, 2, 3], T=3: cửa sổ trượt =", wrong_sliding_at_least([1, 2, 3], 3), "| brute force =", brute_at_least([1, 2, 3], 3), "(vẫn đúng)")
random.seed(11)
for _ in range(5000):
    arr = [random.randint(-4, 4) for _ in range(random.randint(2, 5))]
    T = random.randint(-3, 5)
    if wrong_sliding_at_least(arr, T) != brute_at_least(arr, T):
        print("Có số âm", arr, f"T={T}: cửa sổ trượt =", wrong_sliding_at_least(arr, T), "| brute force =", brute_at_least(arr, T), "(SAI)")
        break

Không âm [1, 2, 3], T=3: cửa sổ trượt = 4 | brute force = 4 (vẫn đúng)
Có số âm [4, 3, 3, 4, -1] T=-1: cửa sổ trượt = 20 | brute force = 15 (SAI)


## Lỗi 2 — Quên trường hợp `k = 0`

Với `k = 0`, đoạn `[v - goal + 1, v - goal - 1]` bị ngược. `bisect_right(hi) - bisect_left(lo)` khi `lo > hi` có thể ra **số âm** (nếu `v - goal` có trong danh sách), làm "gần" âm và đáp án vượt quá tổng số đoạn.

In [13]:
def wrong_no_k0(nums, goal, k):
    n = len(nums)
    seen, close, prefix = [], 0, 0
    for t in range(n + 1):
        if t > 0:
            prefix += nums[t - 1]
        close += bisect_right(seen, prefix - goal + k - 1) - bisect_left(seen, prefix - goal - k + 1)
        insort(seen, prefix)
    return n * (n + 1) // 2 - close


print("Sai :", wrong_no_k0([1, 1, 1], 1, 0), "| Đúng:", distant_bruteforce([1, 1, 1], 1, 0), "(chỉ có 6 đoạn!)")

Sai : 9 | Đúng: 6 (chỉ có 6 đoạn!)


## Lỗi 3 — Sai mút khoảng (dùng `<=` thay `<`)

"Gần" là `|x| < k` (chặt). Nếu lấy đoạn `[v - goal - k, v - goal + k]` sẽ coi nhầm các đoạn có `|tổng - goal| = k` (vốn **xa**) là gần.

In [14]:
def wrong_bounds(nums, goal, k):
    n = len(nums)
    seen, close, prefix = [], 0, 0
    if k == 0:
        return n * (n + 1) // 2
    for t in range(n + 1):
        if t > 0:
            prefix += nums[t - 1]
        close += bisect_right(seen, prefix - goal + k) - bisect_left(seen, prefix - goal - k)   # SAI
        insort(seen, prefix)
    return n * (n + 1) // 2 - close


print("Sai :", wrong_bounds([2, -1, 3], 2, 2), "| Đúng:", distant_bruteforce([2, -1, 3], 2, 2))

Sai : 1 | Đúng: 2


## Lỗi 4 — Quên tiền tố `P[0] = 0`

Bỏ `P[0]` thì mất mọi đoạn bắt đầu từ chỉ số 0.

## Lỗi 5 — Đếm trực tiếp đoạn "xa"

Xa là **hai** tia `(-∞, goal - k]` ∪ `[goal + k, +∞)` — đếm được, nhưng phải truy vấn hai lần và dễ đếm trùng khi `k = 0` (hai tia chồng lên nhau ở `goal`). Đếm phần bù chỉ cần **một** đoạn.

# 12. Mẹo nhận diện

1. "Đếm mảng con có tổng thỏa điều kiện" + **có số âm** → tổng tiền tố + đếm cặp, **không** dùng sliding window.
2. Điều kiện kiểu `|x - goal| >= k` → đếm **phần bù** `|x - goal| < k` (một đoạn liên tục).
3. "Đếm số phần tử đã thấy nằm trong `[lo, hi]`" với giá trị lớn → **nén tọa độ + Fenwick**, hoặc **merge sort**, hoặc `SortedList`.

Các bài liên quan: *Subarray Sum Equals K* (LeetCode 560), *Count of Range Sum* (327), *Count of Smaller Numbers After Self* (315), *Reverse Pairs* (493).

# 13. Biến thể

## Biến thể 1 — Tổng nằm trong `[lo, hi]` (Count of Range Sum)

Đúng là phần "gần" với `lo = goal - k + 1`, `hi = goal + k - 1`. (Bài tập 4.)

## Biến thể 2 — Xa cả hai mục tiêu `g1`, `g2`

Bao hàm – loại trừ: `xa cả hai = tổng - gần(g1) - gần(g2) + gần cả hai`. "Gần cả hai" là giao của hai đoạn — vẫn là **một** đoạn. (Bài tập 6.)

# 14. Template tư duy

```python
# Đếm cặp tiền tố (i < j) với P[i] trong đoạn phụ thuộc P[j]
P = [0] + prefix_sums(nums)
vals = sorted(set(P)); fenwick = Fenwick(len(vals))
for v in P:
    lo, hi = f(v)
    cnt += fenwick.query(bisect_right(vals, hi)) - fenwick.query(bisect_left(vals, lo))
    fenwick.add(bisect_left(vals, v))
```

# 15. Bài tập vận dụng

## Bài 1 — Tính bằng tay

`nums = [1, -1, 1]`, `goal = 0`, `k = 1`. Liệt kê `P`, mô phỏng Cách 2 và tìm đáp án. Kiểm tra bằng cell dưới.

In [15]:
# Kiểm tra Bài 1
print(distant_sorted_list([1, -1, 1], 0, 1, verbose=True))

v=  0 đoạn=[0, 0] seen=[] gần+=0
v=  1 đoạn=[1, 1] seen=[0] gần+=0
v=  0 đoạn=[0, 0] seen=[0, 1] gần+=1
v=  1 đoạn=[1, 1] seen=[0, 0, 1] gần+=1
4


## Bài 2 — Dự đoán output

```python
print(distant_bruteforce([0, 0, 0], 0, 1))
print(distant_bruteforce([0, 0, 0], 0, 0))
print(distant_bruteforce([4], 1, 3))
```

In [16]:
print(distant_bruteforce([0, 0, 0], 0, 1))
print(distant_bruteforce([0, 0, 0], 0, 0))
print(distant_bruteforce([4], 1, 3))

0
6
1


## Bài 3 — Subarray Sum Equals K (LeetCode 560)

Đếm mảng con có tổng **đúng bằng** `target`, `O(n)` bằng từ điển đếm tiền tố.

In [17]:
def exercise_3_subarray_sum(nums: List[int], target: int) -> int:
    # TODO
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_subarray_sum([1, 1, 1], 2) == 2

## Bài 4 — Count of Range Sum (LeetCode 327)

Đếm mảng con có tổng thuộc `[lower, upper]`.

```python
count_range_sum([-2, 5, -1], -2, 2)   -> 3
```

In [18]:
def exercise_4_count_range_sum(nums: List[int], lower: int, upper: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_count_range_sum([-2, 5, -1], -2, 2) == 3

## Bài 5 — Cặp phần tử cách xa

Đếm cặp `(i < j)` với `|nums[i] - nums[j]| >= k`. Gợi ý: thứ tự không quan trọng → sắp xếp + hai con trỏ, `O(n log n)`.

```python
far_pairs([1, 5, 3, 9], 4)   -> 4      # (1,5) (1,9) (5,9) (3,9)
```

In [19]:
def exercise_5_far_pairs(nums: List[int], k: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_far_pairs([1, 5, 3, 9], 4) == 4

## Bài 6 — Mini challenge: xa cả hai mục tiêu

Đếm mảng con có `|tổng - g1| >= k` **và** `|tổng - g2| >= k`, `O(n log n)`.

In [20]:
def exercise_6_distant_both(nums: List[int], g1: int, g2: int, k: int) -> int:
    # TODO
    raise NotImplementedError

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `count[P]` = số lần tiền tố `P` đã xuất hiện, khởi tạo `count[0] = 1`.
- Mức 2: tại tiền tố `v`, cộng `count[v - target]`.
- Mức 3: rồi `count[v] += 1`.

**Bài 4**

- Mức 1: `P[j] - P[i] ∈ [lower, upper]` ⇔ `P[i] ∈ [P[j] - upper, P[j] - lower]`.
- Mức 2: dùng `bisect` + `insort` hoặc Fenwick.
- Mức 3: nhớ tiền tố 0.

**Bài 5**

- Mức 1: sắp tăng dần.
- Mức 2: với mỗi `j`, các `i < j` có `a[i] <= a[j] - k` là một tiền tố.
- Mức 3: `bisect_right(a, a[j] - k, 0, j)`.

**Bài 6**

- Mức 1: viết hàm `close(lo, hi)` đếm mảng con có tổng trong `[lo, hi]`.
- Mức 2: gần `g` ⇔ tổng ∈ `[g - k + 1, g + k - 1]`.
- Mức 3: giao hai đoạn: `[max(lo1, lo2), min(hi1, hi2)]` (rỗng nếu `lo > hi`).

# 16. Đáp án tham khảo

## Đáp án Bài 1

`P = [0, 1, 0, 1]`, `k = 1` → đoạn "gần" của `v` là `[v, v]` (tổng đúng bằng 0).

| v | đoạn | seen trước | gần |
|---:|---|---|---:|
| 0 | `[0, 0]` | `[]` | 0 |
| 1 | `[1, 1]` | `[0]` | 0 |
| 0 | `[0, 0]` | `[0, 1]` | 1 |
| 1 | `[1, 1]` | `[0, 0, 1]` | 1 |

Gần = 2 (`[1, -1]` và `[-1, 1]`) → đáp án `6 - 2 = 4`.

## Đáp án Bài 2

```text
0
6
1
```

`[0, 0, 0]` với `k = 1`: mọi tổng là 0 → không đoạn nào xa. `k = 0`: mọi đoạn xa. `[4]`: `|4 - 1| = 3 >= 3`.

In [21]:
# Đáp án Bài 3
from collections import Counter


def answer_3_subarray_sum(nums: List[int], target: int) -> int:
    count = Counter({0: 1})
    prefix = res = 0
    for x in nums:
        prefix += x
        res += count[prefix - target]
        count[prefix] += 1
    return res


assert answer_3_subarray_sum([1, 1, 1], 2) == 2
assert answer_3_subarray_sum([1, 2, 3], 3) == 2
print("Bài 3: PASS")

Bài 3: PASS


In [22]:
# Đáp án Bài 4
def count_in_range(nums: List[int], lower: int, upper: int) -> int:
    if lower > upper:
        return 0
    seen, prefix, res = [0], 0, 0
    for x in nums:
        prefix += x
        res += bisect_right(seen, prefix - lower) - bisect_left(seen, prefix - upper)
        insort(seen, prefix)
    return res


def answer_4_count_range_sum(nums: List[int], lower: int, upper: int) -> int:
    return count_in_range(nums, lower, upper)


assert answer_4_count_range_sum([-2, 5, -1], -2, 2) == 3
assert answer_4_count_range_sum([0], 0, 0) == 1
print("Bài 4: PASS")

Bài 4: PASS


In [23]:
# Đáp án Bài 5
def answer_5_far_pairs(nums: List[int], k: int) -> int:
    a = sorted(nums)
    return sum(bisect_right(a, a[j] - k, 0, j) for j in range(len(a)))


def brute_far(nums, k):
    return sum(1 for i in range(len(nums)) for j in range(i + 1, len(nums)) if abs(nums[i] - nums[j]) >= k)


assert answer_5_far_pairs([1, 5, 3, 9], 4) == 4
random.seed(5)
for _ in range(500):
    arr = [random.randint(-6, 6) for _ in range(random.randint(1, 8))]
    kk = random.randint(0, 6)
    assert answer_5_far_pairs(arr, kk) == brute_far(arr, kk), (arr, kk)
print("Bài 5: PASS")

Bài 5: PASS


In [24]:
# Đáp án Bài 6
def answer_6_distant_both(nums: List[int], g1: int, g2: int, k: int) -> int:
    n = len(nums)
    total = n * (n + 1) // 2
    if k == 0:
        return total
    lo1, hi1 = g1 - k + 1, g1 + k - 1
    lo2, hi2 = g2 - k + 1, g2 + k - 1
    close1 = count_in_range(nums, lo1, hi1)
    close2 = count_in_range(nums, lo2, hi2)
    both = count_in_range(nums, max(lo1, lo2), min(hi1, hi2))
    return total - close1 - close2 + both


def brute_both(nums, g1, g2, k):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i, n)
               if abs(sum(nums[i:j + 1]) - g1) >= k and abs(sum(nums[i:j + 1]) - g2) >= k)


random.seed(6)
for _ in range(1500):
    arr = [random.randint(-5, 5) for _ in range(random.randint(1, 8))]
    g1, g2, kk = random.randint(-6, 6), random.randint(-6, 6), random.randint(0, 5)
    assert answer_6_distant_both(arr, g1, g2, kk) == brute_both(arr, g1, g2, kk), (arr, g1, g2, kk)
print("Bài 6: PASS (khớp brute force)")

Bài 6: PASS (khớp brute force)


# 17. Tổng kết

**Count Subarrays with Distant Sums**:

1. Tổng đoạn = `P[j] - P[i]`; có số âm nên không dùng hai con trỏ.
2. Đếm phần bù: đoạn **gần** ⇔ `P[i] ∈ [P[j] - goal - k + 1, P[j] - goal + k - 1]`.
3. Đếm "bao nhiêu tiền tố trước đó trong một đoạn giá trị": `bisect + insort`, **Fenwick + nén tọa độ**, hoặc **merge sort**.
4. `k = 0` → mọi đoạn đều xa.

## Độ phức tạp

```text
Time:  O(n log n)
Space: O(n)
```

# 18. Checklist trước khi submit

- [ ] Có tiền tố `P[0] = 0`?
- [ ] Xử lý riêng `k = 0`?
- [ ] Mút đoạn "gần" là `±(k - 1)` (vì `< k` chặt)?
- [ ] Truy vấn trước, chèn sau (chỉ đếm `i < j`)?
- [ ] Đã test: số âm, số lớn `10^9`, `k = 0`, mảng một phần tử?